# 04 — Probe runner (phase B, C1-lin/C2-lin, §7 diagnostics)

Linear-probe sessions on top of finished training runs — the single
frozen recipe of §5.3, features cached once per checkpoint and set.
Three thin sections, each independent:

1. **Phase B (C3/C4):** probe every grid checkpoint
   (`epoch40/50/60.ckpt`) and select the best fused val macro-F1 — the
   operational definition of "plateau" (§6-C3/C4). Writes
   `phase_b_selection.json` in the run folder.
2. **C1-lin / C2-lin:** probe the val-selected `best.ckpt` of a CE run
   (§6 unified linear-probe evaluation).
3. **§7 diagnostics:** AR-set and person probes read against the
   majority baseline — the direct check that GRL removes domain
   information.

Everything here touches **train and val only** — the persisted heads go
to test exclusively in `05_test_final` (§0.7). Each probe costs
minutes; re-running is safe (cached feature files are reused).


## Environment setup

Same conventions as notebook `03`: locate or clone the repo, install
requirements, mount Drive, stage the data zips into `/content/data`
using `configs/paths.yaml`. Checkpoints are read from `ckpt_root` on
Drive; every artifact produced here is written next to them.


In [ ]:
from pathlib import Path
import subprocess
import sys
import torch
import zipfile

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"

# Locate or clone the repository root containing the sharp_har package.
# In Colab, the notebook may run from a temporary working directory.
cwd = Path.cwd().resolve()
if (cwd / "sharp_har").exists():
    REPO_DIR = cwd
elif (cwd.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent
elif (cwd.parent.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent.parent
else:
    REPO_DIR = Path("/content/sharp-har")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# After the clone, so the file actually exists on a fresh runtime.
!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, str(REPO_DIR))

from google.colab import drive
from sharp_har.utils import read_yaml

paths_cfg = read_yaml(REPO_DIR / "configs" / "paths.yaml")
drive_root = Path(paths_cfg["drive_root"])
stage_dir = Path(paths_cfg["stage_dir"])
CKPT_ROOT = Path(paths_cfg["ckpt_root"])

# Mount Drive unconditionally (idempotent): checkpoints live on Drive
# even when the data is already staged on the VM.
drive.mount("/content/drive")

# Stage the zip archives if needed (same convention as 00_setup_smoke).
if not (stage_dir.exists() and any(stage_dir.rglob("*.txt"))):
    stage_dir.mkdir(parents=True, exist_ok=True)
    for zip_name in paths_cfg["zips"]:
        src = drive_root / zip_name
        dst = Path("/content") / zip_name
        print(f"copying {src} -> {dst}")
        subprocess.run(["cp", str(src), str(dst)], check=True)
        with zipfile.ZipFile(dst) as zf:
            zf.extractall(stage_dir)

print("Repo dir:", REPO_DIR)
print("GPU available:", torch.cuda.is_available())
print("Stage dir:", stage_dir)
print("Checkpoint root:", CKPT_ROOT)


## Phase B — grid probe + checkpoint selection (C3, C4)

Runs only after phase A has produced the full pre-committed grid. One
config per session, like notebook `03`. The selection rule is frozen
(§6-C3): best fused val macro-F1 over the grid, ties toward the
earliest epoch. `phase_b_selection.json` records every candidate and is
what `05_test_final` reads — only the selected checkpoint ever goes to
test.


In [ ]:
from sharp_har.probe import select_phase_b

PHASE_B_RUN = "c3_supcon"  # or "c4_supcon_grl"
cfg = read_yaml(REPO_DIR / "configs" / f"{PHASE_B_RUN}.yaml")

selection = select_phase_b(
    CKPT_ROOT / cfg["name"],
    REPO_DIR / cfg["split_file"],
    cfg["train"]["checkpoint_epochs"],
    stage_dir=stage_dir,
    repo_dir=REPO_DIR,
)
print(f"selected: epoch {selection['selected_epoch']} "
      f"(val macro-F1 {selection['selected_val_macro_f1']:.4f})")
selection["candidates"]


## C1-lin / C2-lin — probe on a CE encoder

Same frozen recipe on the val-selected `best.ckpt` of C1 or C2. The
persisted head (`probe_head_best.npz`) is the C1-lin/C2-lin entry of
the main table (§6): representation quality with the classifier held
constant across encoders.


In [ ]:
from sharp_har.probe import probe_encoder

LIN_RUN = "c1_ce"  # or "c2_grl"
cfg = read_yaml(REPO_DIR / "configs" / f"{LIN_RUN}.yaml")

res = probe_encoder(
    CKPT_ROOT / cfg["name"] / "best.ckpt",
    REPO_DIR / cfg["split_file"],
    stage_dir=stage_dir,
    repo_dir=REPO_DIR,
)
{k: res[k] for k in ("best_val_macro_f1", "val_accuracy", "best_epoch", "head_path")}


## §7 diagnostics — AR-set and person probes

Same recipe with `target="ar_set"` or `target="persona"` (person is
derived per sample from the fixed AR-set metadata — no cache change).
Read `val_accuracy` against `val_majority_baseline` (NOT 1/n — classes
are unbalanced). Expected: high in C1/C3, toward the baseline in C2/C4
— the key invariance evidence of §9. The person probe is qualitative
only (person and environment are confounded in the AR-sets, §7).

For C3/C4 point at the phase-B **selected** checkpoint (from
`phase_b_selection.json`), not at an arbitrary grid epoch.


In [ ]:
from sharp_har.probe import probe_encoder

P2_SPLIT = REPO_DIR / "splits" / "p2_lab.json"
DIAG_CHECKPOINTS = {
    "C1": CKPT_ROOT / "C1" / "best.ckpt",
    "C2": CKPT_ROOT / "C2" / "best.ckpt",
    # "C3": Path(...),  # phase-B selected checkpoint, see phase_b_selection.json
    # "C4": Path(...),
}

for label, ckpt in DIAG_CHECKPOINTS.items():
    for target in ("ar_set", "persona"):
        res = probe_encoder(ckpt, P2_SPLIT, target=target,
                            stage_dir=stage_dir, repo_dir=REPO_DIR)
        print(f"{label} [{target}]: val accuracy {res['val_accuracy']:.3f} "
              f"vs majority baseline {res['val_majority_baseline']:.3f}")


## Archiving

As for every executed session: download the notebook **with outputs**
and commit it verbatim under `notebooks/runs/` (e.g.
`2026-07-19_phase_b_c3.ipynb`); this template stays output-free on Git.
The probe artifacts (feature caches, heads, JSON summaries) stay on
Drive next to the checkpoints — the JSON summaries are small and are
the numbers quoted in the report.
